# 08-02 · 첫 회귀 (모형 사다리 M1~M3, 이벤트 스터디, 강건성 R1~R4)

| 항목 | 내용 |
|---|---|
| 하는 일 | 사전 고정한 모형들을 **한 번** 실행: ICR 변화가 사전 단기차입(E)이 클수록 인상기에 더 커졌는지(E×Tight 계수) |
| 읽는 파일 | `outputs/07_분석/패널_기업분기.csv` |
| 만드는 파일 | `outputs/07_분석/회귀_결과.csv`, `회귀_요약.txt`, `이벤트스터디.csv`, `이벤트스터디.png` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 12초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

**모형** — `ICR_it = α_i + λ(...) + β·(E_i × Tight_t) [+ γ'·사전특성_i×Tight_t] + ε_it`

- M1: 기업 FE + 분기 FE / M2: 기업 FE + 업종(앞 2자리)×분기 FE / M3: M2 + 사전특성×Tight
- 표준오차: 기업 단위 군집(CR1), 검정은 t(G−1). 싱글톤은 반복 제거.
- 고정효과는 직접 구현한 교대 투영으로 흡수 — 구현 확인은 `03_회귀_구현검증`.

규칙은 회귀 전에 고정했고 **결과를 보고 사양을 고르거나 바꾸지 않았습니다.**

> **성격** — 이 단계의 결과는 모두 **사후 탐색**입니다(결과를 본 뒤 설계가 바뀔 수 있고, 팀이 규칙을 확정하기 전입니다). '신호 기준'은 에이전트가 정한 임의 기준이며 확인 증거가 아닙니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""회귀: 사전 고정한 모형 사다리(M1~M3), 이질성, 이벤트 스터디, 보조 결과, 강건성 R1~R4, 진단을 **한 번** 실행한다.
규칙은 docs/A_결정_제안_근거표_11-05.md 8절(회귀 전 고정). 결과를 보고 사양을 고르거나 바꾸지 않는다.

모형: ICR_it = α_i + λ(...) + β·(E_i × Tight_t) + [γ'·사전특성_i×Tight_t] + ε_it
  M1: 기업 FE + 분기 FE / M2: 기업 FE + 업종(앞 2자리)×분기 FE / M3: M2 + 사전특성×Tight
  표준오차: 기업 단위 군집(CR1: G/(G-1)·(N-1)/(N-K), K에는 기업 FE는 군집에 포함되므로 세지 않고 그 밖의 FE는 셈), 검정은 t(G-1)
  싱글톤(FE 칸에 관측 1개)은 반복해서 제거.
고정효과는 직접 구현한 교대 투영(demeaning)으로 흡수한다(전용 라이브러리 없음). 구현 검증은 03_회귀_구현검증.py.
"""
import sys
from pathlib import Path
import matplotlib
import numpy as np
import pandas as pd
from scipy import stats
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
OUT = config.BASE / "outputs" / "07_분석"
BASE_T = 9           # 2021Q2 (t=0이 2019Q1)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

#### `demean()`

In [3]:
def demean(M, codes, maxit=1000, tol=1e-10):
    M = M.astype(float).copy()
    cnts = [np.bincount(c) for c in codes]
    for _ in range(maxit):
        mx = 0.0
        for c, cn in zip(codes, cnts):
            for j in range(M.shape[1]):
                m = np.bincount(c, weights=M[:, j]) / cn
                M[:, j] -= m[c]
                mx = max(mx, np.abs(m).max())
        if mx < tol * max(1.0, np.abs(M).max()):
            break
    return M

#### `fe_ols()`

In [4]:
def fe_ols(df, y, xs, fes, cluster="corp_code", label=""):
    cols = [y] + xs + fes + [cluster]
    d = df[list(dict.fromkeys(cols))].dropna().copy()
    n0 = len(d)
    while True:
        keep = np.ones(len(d), bool)
        for f in fes:
            keep &= d[f].map(d[f].value_counts()).values > 1
        if keep.all():
            break
        d = d[keep]
    codes = [pd.factorize(d[f])[0] for f in fes]
    M = d[[y] + xs].to_numpy(float)
    Mt = demean(M, codes)
    yt, Xt = Mt[:, 0], Mt[:, 1:]
    XtX = Xt.T @ Xt
    beta = np.linalg.solve(XtX, Xt.T @ yt)
    e = yt - Xt @ beta
    cl = pd.factorize(d[cluster])[0]
    G = cl.max() + 1
    S = pd.DataFrame(Xt * e[:, None]).groupby(cl).sum().to_numpy()
    bread = np.linalg.inv(XtX)
    N, k = Xt.shape
    nonnested = sum(d[f].nunique() for f in fes if f != cluster and f != "corp_code")
    K = k + nonnested
    c = G / (G - 1) * (N - 1) / (N - K)
    V = c * bread @ (S.T @ S) @ bread
    se = np.sqrt(np.diag(V))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), G - 1)
    r2_within = 1 - (e @ e) / (yt @ yt)
    return {"label": label, "y": y, "xs": xs, "beta": beta, "se": se, "t": t, "p": p, "V": V, "N": N, "N0": n0, "G": G,
            "r2w": r2_within, "dropped_singletons": n0 - N}

#### `fmt()`

In [5]:
def fmt(res, i=0):
    return f"{res['beta'][i]: .4f} ({res['se'][i]:.4f}) p={res['p'][i]:.3f}"

#### `main()`

이 함수(152줄) 안의 단계 — 코드 주석에서 뽑은 목차:
1. ── 모형 사다리 ──
2. ── 이질성 ──
3. ── 이벤트 스터디 ──
4. ── 보조 결과 ──
5. ── 강건성 ──
6. 규칙 ④ 보조: 애매=0
7. ── 진단 ──

In [6]:
def main():
    P = pd.read_csv(OUT / "패널_기업분기.csv", dtype={"corp_code": str})
    P["corp_code"] = P["corp_code"].str.zfill(8)
    P["ind_t"] = P["업종2"].astype(str) + "_" + P["t"].astype(str)
    P["Ex"] = P["E_pre_A"] * P["tight"]
    P["ExB"] = P["E_pre_B"] * P["tight"]
    P["Ex21"] = P["E_2021Q2"] * P["tight"]
    P["Ex0"] = P["E_pre_A_amb0"] * P["tight"]
    P["size_x"] = P["ln_assets_pre"] * P["tight"]
    P["lev_x"] = P["lev_pre"] * P["tight"]
    P["roa_x"] = P["roa_pre"] * P["tight"]
    L = []
    rows = []

    def rec(res, spec, var="E×Tight", i=0):
        rows.append({"사양": spec, "종속변수": res["y"], "변수": var, "계수": res["beta"][i], "표준오차": res["se"][i], "t": res["t"][i],
                     "p": res["p"][i], "관측": res["N"], "기업": res["G"], "within_R2": res["r2w"]})

    S = P[P["표본_주"]].copy()
    firms_main = S["corp_code"].nunique()
    L.append(f"주 표본: 기업 {firms_main:,}곳, 기업-분기 {len(S):,}개 (ICR 관측 {int(S['icr'].notna().sum()):,}개)")
    lo, hi = S["icr"].quantile(.01), S["icr"].quantile(.99)
    S["icr_w"] = S["icr"].clip(lo, hi)
    L.append(f"ICR 1%·99% 절단 경계: {lo:.2f} / {hi:.2f}")
    e1 = S.drop_duplicates("corp_code")["E_pre_A"]
    L.append(f"E_pre_A 분포(주 표본 기업): 평균 {e1.mean():.3f}, 중앙값 {e1.median():.3f}, 표준편차 {e1.std():.3f}, 상위25% 경계 {e1.quantile(.75):.3f}")
    L.append(f"긴축 3%p, E 0.1 증가일 때 계수 해석 배율: 0.1×3.0 = 0.30 (예: β=-1이면 ICR 약 -0.3)")
    L.append("")

    # ── 모형 사다리 ──
    L.append("[1] 모형 사다리 — 종속변수 ICR (E×Tight 계수, 기업 군집 표준오차)")
    m1 = fe_ols(S, "icr", ["Ex"], ["corp_code", "t"], label="M1")
    m2 = fe_ols(S, "icr", ["Ex"], ["corp_code", "ind_t"], label="M2")
    m3 = fe_ols(S, "icr", ["Ex", "size_x", "lev_x", "roa_x"], ["corp_code", "ind_t"], label="M3")
    for nm, m in (("M1 기업FE+분기FE", m1), ("M2 기업FE+업종×분기FE (기준)", m2)):
        L.append(f"  {nm:34s} β={fmt(m)}  N={m['N']:,} 기업={m['G']:,} 싱글톤제거={m['dropped_singletons']}")
        rec(m, nm)
    L.append(f"  {'M3 M2+사전특성×Tight':34s} β={fmt(m3)}  N={m3['N']:,} 기업={m3['G']:,}")
    rec(m3, "M3 M2+사전특성×Tight")
    for j, v in enumerate(["ln자산_사전×Tight", "레버리지_사전×Tight", "영업이익/자산_사전×Tight"], 1):
        L.append(f"       {v:28s} {fmt(m3, j)}")
        rec(m3, "M3 M2+사전특성×Tight", v, j)
    m2w = fe_ols(S, "icr_w", ["Ex"], ["corp_code", "ind_t"], label="M2 절단")
    L.append(f"  {'M2, ICR 1%·99% 절단(보조)':34s} β={fmt(m2w)}  N={m2w['N']:,}")
    rec(m2w, "M2 ICR 절단(보조)")
    L.append("")

    # ── 이질성 ──
    L.append("[2] 이질성 — 사전 평균 자산 3분위별 β (M2)")
    q1, q2 = S.drop_duplicates("corp_code")["ln_assets_pre"].quantile([1 / 3, 2 / 3])
    S["tercile"] = np.where(S["ln_assets_pre"] <= q1, 1, np.where(S["ln_assets_pre"] <= q2, 2, 3))
    for k in (1, 2, 3):
        S[f"Ex_s{k}"] = S["Ex"] * (S["tercile"] == k)
    mh = fe_ols(S, "icr", ["Ex_s1", "Ex_s2", "Ex_s3"], ["corp_code", "ind_t"], label="이질성")
    for k, nm in enumerate(["하위 1/3(작은 기업)", "중간 1/3", "상위 1/3(큰 기업)"]):
        L.append(f"  {nm:20s} β={fmt(mh, k)}")
        rec(mh, "이질성(자산 3분위)", f"E×Tight, {nm}", k)
    L.append("")

    # ── 이벤트 스터디 ──
    L.append("[3] 이벤트 스터디 — 분기별 E 계수, 기준 2021Q2(t=9), M2의 FE")
    ts = [t for t in range(0, 23) if t != BASE_T]
    for t in ts:
        S[f"Ev{t}"] = S["E_pre_A"] * (S["t"] == t)
    me = fe_ols(S, "icr", [f"Ev{t}" for t in ts], ["corp_code", "ind_t"], label="이벤트스터디")
    names = {t: f"{2019 + t // 4}Q{t % 4 + 1}" for t in range(23)}
    ev = []
    for i, t in enumerate(ts):
        ev.append({"분기": names[t], "t": t, "계수": me["beta"][i], "표준오차": me["se"][i], "p": me["p"][i]})
        L.append(f"  {names[t]}  {me['beta'][i]: .3f} ({me['se'][i]:.3f})")
    pre_idx = [i for i, t in enumerate(ts) if t < BASE_T]
    b = me["beta"][pre_idx]
    Vp = me["V"][np.ix_(pre_idx, pre_idx)]
    W = float(b @ np.linalg.solve(Vp, b))
    pj = stats.f.sf(W / len(pre_idx), len(pre_idx), me["G"] - 1)
    L.append(f"  사전 분기 계수 {len(pre_idx)}개 공동 0 검정(군집 Wald, F 근사): W={W:.2f}, p={pj:.3f}")
    pd.DataFrame(ev).to_csv(OUT / "이벤트스터디.csv", index=False, encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    xs = [e["t"] for e in ev]
    ax.errorbar(xs, [e["계수"] for e in ev], yerr=[1.96 * e["표준오차"] for e in ev], fmt="o", capsize=3, color="#165a70")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(BASE_T + 0.5, color="red", ls="--", lw=0.8)
    ax.set_xticks(range(23))
    ax.set_xticklabels([names[t] for t in range(23)], rotation=70, fontsize=8)
    ax.set_ylabel("E 계수 (기준: 2021Q2)")
    ax.set_title("이벤트 스터디: 사전 단기차입 노출도(E)와 ICR (M2, 95% 신뢰구간, 기업 군집)")
    fig.tight_layout()
    fig.savefig(OUT / "이벤트스터디.png", dpi=150)
    L.append("")

    # ── 보조 결과 ──
    L.append("[4] 보조 결과 (M2, E×Tight)")
    for y, nm in (("fc_a", "금융비용 TTM/자산"), ("oi_a", "영업이익 TTM/자산")):
        m = fe_ols(S, y, ["Ex"], ["corp_code", "ind_t"], label=nm)
        L.append(f"  {nm:22s} β={fmt(m)}  N={m['N']:,}")
        rec(m, f"보조결과 {nm}")
    L.append("")

    # ── 강건성 ──
    L.append("[5] 강건성 (M2, 종속변수 ICR, E×Tight 계수)")
    # R1
    R1 = S[S["E_pre_B"].notna()]
    r1 = fe_ols(R1, "icr", ["ExB"], ["corp_code", "ind_t"], label="R1")
    L.append(f"  R1 E=단기차입/총차입금          β={fmt(r1)}  N={r1['N']:,} 기업={r1['G']:,}")
    rec(r1, "R1 E=단기차입/총차입금", "E_B×Tight")
    # R2
    R2 = P[(P["표본_주"]) & (P["post_icr_cash_n"] >= 8)]
    r2 = fe_ols(R2, "icr_cash", ["Ex"], ["corp_code", "ind_t"], label="R2")
    L.append(f"  R2 ICR 분모=현금 이자지급       β={fmt(r2)}  N={r2['N']:,} 기업={r2['G']:,}")
    rec(r2, "R2 ICR분모=현금 이자지급")
    # R3
    R3 = S[S["E_2021Q2"].notna()]
    r3 = fe_ols(R3, "icr", ["Ex21"], ["corp_code", "ind_t"], label="R3")
    L.append(f"  R3 E=2021Q2 단독                β={fmt(r3)}  N={r3['N']:,} 기업={r3['G']:,}")
    rec(r3, "R3 E=2021Q2", "E_2021Q2×Tight")
    # R4
    R4 = P[P["표본_R4"]].copy()
    R4["ind_t"] = R4["업종2"].astype(str) + "_" + R4["t"].astype(str)
    r4 = fe_ols(R4, "icr", ["Ex"], ["corp_code", "ind_t"], label="R4")
    L.append(f"  R4 사전기간 혼재만 제외         β={fmt(r4)}  N={r4['N']:,} 기업={r4['G']:,}")
    rec(r4, "R4 사전기간 혼재만 제외")
    # 규칙 ④ 보조: 애매=0
    S0 = S[S["E_pre_A_amb0"].notna()]
    s0 = fe_ols(S0, "icr", ["Ex0"], ["corp_code", "ind_t"], label="S1")
    L.append(f"  S1 애매 분기를 E=0으로(규칙④ 보조) β={fmt(s0)}  N={s0['N']:,} 기업={s0['G']:,}")
    rec(s0, "S1 애매=0 민감도", "E_amb0×Tight")
    L.append("")

    # ── 진단 ──
    L.append("[6] 진단")
    F = S.drop_duplicates("corp_code")
    dm = pd.get_dummies(F["업종2"], dtype=float)
    X = dm.to_numpy()
    yv = F["E_pre_A"].to_numpy()
    fit = X @ np.linalg.lstsq(X, yv, rcond=None)[0]
    r2 = 1 - ((yv - fit) ** 2).sum() / ((yv - yv.mean()) ** 2).sum()
    L.append(f"  (1) 업종 더미로 설명되는 E의 비율 R²={r2:.3f}; 업종 안에서 남는 E의 표준편차 {np.std(yv - fit):.3f} (전체 {yv.std():.3f})")
    cm = F[["E_pre_A", "ln_assets_pre", "lev_pre", "roa_pre"]].corr()
    L.append("  (3) 기업 단위 상관(E, ln자산_사전, 레버리지_사전, 영업이익/자산_사전):")
    for a in cm.index:
        L.append("      " + f"{a:14s} " + " ".join(f"{cm.loc[a, b]: .2f}" for b in cm.columns))
    ic = S["icr"].dropna()
    L.append(f"  (4) ICR 극단값: >50 {int((ic > 50).sum()):,}개, >100 {int((ic > 100).sum()):,}개, <-20 {int((ic < -20).sum()):,}개, 최대 {ic.max():.1f}, 최소 {ic.min():.1f}")
    top = S.loc[S["icr"].abs().nlargest(5).index, ["corp_code", "year", "q", "icr"]]
    L.append("      |ICR| 상위 5: " + "; ".join(f"{r.corp_code} {int(r.year)}Q{int(r.q)} {r.icr:.1f}" for r in top.itertuples()))
    L.append("  (5) 기업 수·관측: 위 사양별 N, 기업 수 및 패널 요약(패널_요약.txt)")
    L.append("  (6) 위약 검정(가짜 긴축 시점)은 이번 실행에 넣지 않음(8절 기재).")
    text = "\n".join(L)
    print(text)
    (OUT / "회귀_요약.txt").write_text(text, encoding="utf-8")
    pd.DataFrame(rows).to_csv(OUT / "회귀_결과.csv", index=False, encoding="utf-8-sig")
    print("저장:", OUT)

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [7]:
main()

주 표본: 기업 789곳, 기업-분기 18,147개 (ICR 관측 15,768개)
ICR 1%·99% 절단 경계: -48.84 / 491.79
E_pre_A 분포(주 표본 기업): 평균 0.158, 중앙값 0.139, 표준편차 0.128, 상위25% 경계 0.234
긴축 3%p, E 0.1 증가일 때 계수 해석 배율: 0.1×3.0 = 0.30 (예: β=-1이면 ICR 약 -0.3)

[1] 모형 사다리 — 종속변수 ICR (E×Tight 계수, 기업 군집 표준오차)
  M1 기업FE+분기FE                       β=-73.5619 (281.9737) p=0.794  N=15,768 기업=789 싱글톤제거=0
  M2 기업FE+업종×분기FE (기준)               β=-11.8421 (267.5159) p=0.965  N=15,757 기업=789 싱글톤제거=11
  M3 M2+사전특성×Tight                   β=-229.7043 (285.9535) p=0.422  N=15,696 기업=784
       ln자산_사전×Tight                 13.4334 (10.0173) p=0.180
       레버리지_사전×Tight                 182.1298 (240.9173) p=0.450
       영업이익/자산_사전×Tight             -295.9814 (2377.0832) p=0.901
  M2, ICR 1%·99% 절단(보조)              β= 11.4285 (4.7897) p=0.017  N=15,757

[2] 이질성 — 사전 평균 자산 3분위별 β (M2)
  하위 1/3(작은 기업)        β=-22.4528 (308.5666) p=0.942
  중간 1/3               β= 11.4897 (238.1417) p=0.962
  상위 1/3(큰 기업)         β=-21.1754 (244.3458) p=0.931

[3] 

### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [8]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/07_분석/회귀_결과.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

회귀_결과.csv | 17행 × 10열


,사양,종속변수,변수,계수,표준오차,t,p,관측,기업,within_R2
0,M1 기업FE+분기FE,icr,E×Tight,-73.561885,281.973711,-0.260882,0.794252,15768,789,1.406294e-05
1,M2 기업FE+업종×분기FE (기준),icr,E×Tight,-11.842093,267.515948,-0.044267,0.964703,15757,789,3.424814e-07
2,M3 M2+사전특성×Tight,icr,E×Tight,-229.704254,285.953490,-0.803292,0.422050,15696,784,9.431341e-05
3,M3 M2+사전특성×Tight,icr,ln자산_사전×Tight,13.433383,10.017348,1.341012,0.180305,15696,784,9.431341e-05
4,M3 M2+사전특성×Tight,icr,레버리지_사전×Tight,182.129831,240.917334,0.755985,0.449886,15696,784,9.431341e-05
